# 🤖 Intro to Machine Learning: Classification, Regression & Clustering

Welcome! This notebook is **interactive** — instead of just reading code and looking at static plots, you'll get sliders and dropdowns to play with real models and watch them change in real time.

**How to use this notebook:**
1. Run the cells **top to bottom** (Shift + Enter).
2. Wherever you see a slider or dropdown, drag it / change it and watch the plot update.
3. Look for **🤔 Quick Check** boxes — click to reveal the answer after you've thought about it.
4. Don't be afraid to break things! Push sliders to extreme values and see what happens — that's how you learn.

We'll cover three core ML tasks:

| Task | Question it answers |
|---|---|
| 🔍 Classification | "What **category** is this?" |
| 📈 Regression | "**How much** / what number?" |
| 🎯 Clustering | "Who **belongs together**?" |


## 📦 Setup & Imports

Run this cell first. It installs the interactive widgets library (if you don't have it) and imports everything we'll use.

In [ ]:
# %pip install matplotlib seaborn scikit-learn pandas numpy ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import make_classification, make_regression, make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score

from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
from IPython.display import display, Markdown

# Visual style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5.5)

print("✅ All set! Scroll down to get started.")

---
# 🔍 Part 1: Classification — "What Category?"

### 1.1 Generate a Classification Dataset

Let's create a synthetic 2-class dataset so we can literally *see* how classification works in 2D.

In [ ]:
# Generate synthetic classification data
X, y = make_classification(
    n_samples=300,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    random_state=42
)

df_class = pd.DataFrame(X, columns=['Feature 1', 'Feature 2'])
df_class['Target'] = y

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

plt.figure(figsize=(9, 5.5))
sns.scatterplot(
    data=df_class, x='Feature 1', y='Feature 2',
    hue='Target', style='Target', palette=['#FF6B6B', '#4ECDC4'],
    s=80, alpha=0.7
)
plt.title('📊 Classification Dataset: Two Classes')
plt.legend(title='Class', labels=['Class 0', 'Class 1'])
plt.show()

📌 **Observation:** Can you see roughly where the two classes separate? The goal of classification is to find a **boundary** that divides them.

### 1.2 Interactive: Train a KNN Classifier

Drag the slider below to change **K** (the number of neighbors KNN looks at when making a prediction) and watch the decision boundary and accuracy update live.

In [ ]:
def plot_knn(k=5):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05), np.arange(y_min, y_max, 0.05))
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    plt.figure(figsize=(9, 5.5))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='coolwarm', edgecolors='k', s=70)
    plt.title(f'📊 KNN Decision Boundary (K={k})  |  Test Accuracy: {acc:.1%}')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

interact(plot_knn, k=IntSlider(min=1, max=100, step=1, value=5, description='K (neighbors)'));

🤔 **Quick Check:** What happens to the boundary when K is very small (like 1) vs. very large (like 100)?

<details>
<summary>Click to reveal answer</summary>

With **K=1**, the model only looks at the single closest point, so the boundary becomes very jagged and "memorizes" the training data — this is **overfitting**. With a **very large K**, the model averages over so many neighbors that the boundary becomes overly smooth (even a straight-ish line) and can underfit, ignoring real structure in the data. There's usually a sweet spot in between.
</details>


### 1.3 Confusion Matrix

A confusion matrix gives a more detailed picture than accuracy alone — it shows exactly *which* mistakes the model makes.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred 0', 'Pred 1'],
            yticklabels=['True 0', 'True 1'])
plt.title('📋 Confusion Matrix - KNN (K=5)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

📌 **Reading it:**
- **Top-left (TN):** correctly predicted Class 0
- **Top-right (FP):** predicted Class 1, actually Class 0
- **Bottom-left (FN):** predicted Class 0, actually Class 1
- **Bottom-right (TP):** correctly predicted Class 1

### 1.4 🧪 Try It Yourself — Compare Classifiers

Use the dropdown to switch between three different classifiers and see how their decision boundaries differ in *shape*.

In [ ]:
def compare_classifiers(model_name='KNN'):
    if model_name == 'KNN':
        model = KNeighborsClassifier(n_neighbors=5)
    elif model_name == 'Decision Tree':
        model = DecisionTreeClassifier(max_depth=3, random_state=42)
    else:
        model = LogisticRegression()

    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05), np.arange(y_min, y_max, 0.05))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    plt.figure(figsize=(9, 5.5))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='coolwarm', edgecolors='k', s=70)
    plt.title(f'📊 {model_name} Decision Boundary  |  Test Accuracy: {acc:.1%}')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

interact(compare_classifiers, model_name=Dropdown(
    options=['KNN', 'Decision Tree', 'Logistic Regression'], value='KNN', description='Model:'
));

💡 **Discussion Question:** Why does the Decision Tree have a *blocky*, rectangular boundary while Logistic Regression draws a single *straight* line, and KNN draws a *smooth curve*? What does that tell you about how each algorithm "thinks"?

---
# 📈 Part 2: Regression — "How Much?"

### 2.1 Generate a Regression Dataset

This time, our target isn't a category — it's a **continuous number**.

In [ ]:
X_reg, y_reg = make_regression(n_samples=200, n_features=1, noise=15, random_state=42)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

plt.figure(figsize=(9, 5.5))
plt.scatter(X_reg, y_reg, alpha=0.6, color='#4A90D9', s=60)
plt.title('📈 Regression Dataset')
plt.xlabel('Feature (X)')
plt.ylabel('Target (Y)')
plt.show()

📌 **Observation:** We want to predict a Y value for any given X — not put it in a category.

### 2.2 Interactive: Polynomial Regression

A plain straight line (degree 1) is **Linear Regression**. Increasing the "degree" lets the curve bend. Drag the slider and watch what happens to the fit — and to R² — as the degree increases.

In [ ]:
def plot_poly(degree=1):
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree)),
        ('lin_reg', LinearRegression())
    ])
    model.fit(X_train_reg, y_train_reg)
    y_pred = model.predict(X_test_reg)

    mse = mean_squared_error(y_test_reg, y_pred)
    r2 = r2_score(y_test_reg, y_pred)

    X_line = np.linspace(X_reg.min(), X_reg.max(), 200).reshape(-1, 1)
    y_line = model.predict(X_line)

    plt.figure(figsize=(9, 5.5))
    plt.scatter(X_train_reg, y_train_reg, alpha=0.5, label='Training Data', color='#4A90D9')
    plt.scatter(X_test_reg, y_test_reg, alpha=0.5, label='Test Data', color='#FF6B6B')
    plt.plot(X_line, y_line, 'k-', linewidth=2, label=f'Degree {degree} Fit')
    plt.ylim(y_reg.min() - 40, y_reg.max() + 40)
    plt.title(f'📈 Polynomial Regression (Degree {degree})  |  Test R²: {r2:.1%}, MSE: {mse:.1f}')
    plt.xlabel('Feature (X)')
    plt.ylabel('Target (Y)')
    plt.legend()
    plt.show()

interact(plot_poly, degree=IntSlider(min=1, max=15, step=1, value=1, description='Degree'));

🤔 **Quick Check:** Push the degree slider up to 12-15. The line starts wiggling wildly at the edges. Is that a *better* model?

<details>
<summary>Click to reveal answer</summary>

No! Even though a high-degree polynomial can fit the **training** data almost perfectly, watch the **test R²** — it usually gets *worse*, not better, at high degrees. The model is **overfitting**: memorizing noise in the training data instead of learning the true underlying pattern. This is one of the most important ideas in machine learning.
</details>

---
# 🎯 Part 3: Clustering — "Who Belongs Together?"

### 3.1 Generate a Clustering Dataset

Notice: there are **no labels** here. The algorithm has to discover groups on its own — this is called *unsupervised learning*.

In [ ]:
X_clust, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=42)

plt.figure(figsize=(9, 5.5))
plt.scatter(X_clust[:, 0], X_clust[:, 1], alpha=0.6, s=60, color='#9B59B6')
plt.title('🎯 Unlabeled Data – Can you see the groups?')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

### 3.2 Interactive: K-Means Clustering

Drag K and watch K-Means try to find that many groups. Try setting K way too low or too high!

In [ ]:
def plot_kmeans(k=4):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_clust)
    labels = kmeans.labels_
    centroids = kmeans.cluster_centers_

    plt.figure(figsize=(9, 5.5))
    plt.scatter(X_clust[:, 0], X_clust[:, 1], c=labels, cmap='viridis', alpha=0.6, s=60)
    plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='X', s=200,
                edgecolors='black', linewidths=2, label='Centroids')
    plt.title(f'🎯 K-Means Clustering Results (K={k})  |  Inertia: {kmeans.inertia_:.0f}')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.legend()
    plt.show()

interact(plot_kmeans, k=IntSlider(min=1, max=10, step=1, value=4, description='K (clusters)'));

📌 **Observation:** Each color is a cluster; the red X's are the centroids (cluster centers). Notice how the "wrong" K either splits a real group in two, or merges two real groups into one.

### 3.3 How Do We Pick K? The Elbow Method

In [ ]:
inertias = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_clust)
    inertias.append(km.inertia_)

plt.figure(figsize=(9, 5.5))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.title('📊 Elbow Method – Finding Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.xticks(list(K_range))
plt.grid(True, alpha=0.3)
plt.show()

📌 **Observation:** Look for the "elbow" — where inertia stops dropping sharply. Here it's around K=4, which matches the 4 real groups we generated.

### 3.4 🧪 Bonus: When K-Means Struggles

K-Means assumes clusters are round "blobs." Use the dropdown to compare it against **DBSCAN**, a density-based algorithm, on a trickier "moons" shaped dataset.

In [ ]:
X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)

def compare_clustering(algorithm='K-Means'):
    if algorithm == 'K-Means':
        model = KMeans(n_clusters=2, random_state=42, n_init=10)
        labels = model.fit_predict(X_moons)
    else:
        model = DBSCAN(eps=0.2, min_samples=5)
        labels = model.fit_predict(X_moons)

    plt.figure(figsize=(9, 5.5))
    plt.scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap='coolwarm', alpha=0.7, s=60)
    plt.title(f'🌙 {algorithm} on Moon-Shaped Data')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

interact(compare_clustering, algorithm=Dropdown(
    options=['K-Means', 'DBSCAN'], value='K-Means', description='Algorithm:'
));

📌 **Observation:** K-Means cuts the moons in half with a straight-ish boundary because it only understands "distance to a center." DBSCAN follows the *density* of points and correctly traces each crescent shape.

---
## 📝 Summary – Which One to Use?

In [ ]:
summary_data = {
    'Task Type': ['Classification', 'Regression', 'Clustering'],
    'Question': ['What category?', 'How much?', 'Who belongs together?'],
    'Output Type': ['Discrete (Class)', 'Continuous (Number)', 'No Labels (Groups)'],
    'Example': ['Spam/Not Spam', 'House Price', 'Customer Segments'],
    'Algorithms': ['KNN, Decision Tree, Logistic Regression', 'Linear, Polynomial', 'K-Means, DBSCAN']
}

summary_df = pd.DataFrame(summary_data)
summary_df

## 💭 Final Challenge: Think About These Questions

1. You have a dataset with 100,000 customer records and **no labels**. You want to group them into marketing personas. Which task is this?

2. You want to predict whether a mortgage application will be **approved or denied**. Classification or Regression?

3. What if instead you want to predict the **exact interest rate** the bank will offer? Now which task is it?

4. Look back at the Elbow Method graph. If the graph had **no clear bend**, what might that mean about your data?

<details>
<summary>🔑 Click to reveal all answers</summary>

1. **Clustering** — no labels, discovering natural groups is unsupervised learning.
2. **Classification** — the output is one of two discrete categories (approved / denied).
3. **Regression** — an exact interest rate is a continuous number, not a category.
4. It might mean the data doesn't have clearly separated groups — the "natural" number of clusters is ambiguous, or the data doesn't cluster well at all (e.g., it's more uniformly spread out).
</details>
